# `mdpp` Example: Projection Analysis and Ramachandran

PCA/TICA visualization, scree plots, cumulative variance, projection scatter,
Ramachandran diagram.


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
from mplplots.utils import auto_ticks

from mdpp.analysis.decomposition import (
    compute_pca,
    compute_tica,
    featurize_backbone_torsions,
)
from mdpp.core.trajectory import align_trajectory, load_trajectory
from mdpp.plots.scatter import (
    plot_pca_cumulative_variance,
    plot_pca_scree,
    plot_projection,
    plot_ramachandran,
)

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5
TICA_LAG_FRAMES = 20

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)
traj = align_trajectory(traj, atom_selection="name CA")

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Featurize and Project


In [ ]:
torsions_sincos = featurize_backbone_torsions(traj, sincos_embedding=True)
torsions_raw = featurize_backbone_torsions(traj, sincos_embedding=False)

pca = compute_pca(torsions_sincos.values, n_components=10, standardize=True)
tica = compute_tica(torsions_sincos.values, lagtime=TICA_LAG_FRAMES, n_components=2)

print(f"PCA cumulative variance: {pca.explained_variance_ratio.sum():.1%}")

## Scree Plot and Cumulative Variance


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
plot_pca_scree(pca, ax=axes[0])
axes[0].set_title("PCA Scree Plot")
plot_pca_cumulative_variance(pca, ax=axes[1])
axes[1].set_title("Cumulative Variance")
auto_ticks(*axes)
fig.tight_layout()

## PCA and TICA Projections


In [ ]:
import numpy as np

time = np.arange(traj.n_frames, dtype=np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=120)
plot_projection(pca, ax=axes[0], color_by=time, cmap="viridis", s=3)
axes[0].set_title("PCA Projection")
plot_projection(tica, ax=axes[1], color_by=time, cmap="viridis", s=3)
axes[1].set_title("TICA Projection")
fig.tight_layout()

## Ramachandran Diagram


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
plot_ramachandran(torsions_raw, ax=ax)
ax.set_title("Ramachandran Diagram")
fig.tight_layout()